##### Copyright 2025 Google LLC.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.


# vrp_time_windows

<table align="left">
<td>
<a href="https://colab.research.google.com/github/google/or-tools/blob/main/examples/notebook/routing/vrp_time_windows.ipynb"><img src="https://raw.githubusercontent.com/google/or-tools/main/tools/colab_32px.png"/>Run in Google Colab</a>
</td>
<td>
<a href="https://github.com/google/or-tools/blob/main/ortools/routing/samples/vrp_time_windows.py"><img src="https://raw.githubusercontent.com/google/or-tools/main/tools/github_32px.png"/>View source on GitHub</a>
</td>
</table>

First, you must install [ortools](https://pypi.org/project/ortools/) package in this colab.

In [ ]:
%pip install ortools


Vehicles Routing Problem (VRP) with Time Windows.


In [ ]:
from typing import Any, Dict

from ortools.constraint_solver.python import constraint_solver
from ortools.routing import enums_pb2, parameters_pb2
from ortools.routing.python import routing



def create_data_model() -> Dict[str, Any]:
    """Stores the data for the problem."""
    data = {}
    data["time_matrix"] = [
        [0, 6, 9, 8, 7, 3, 6, 2, 3, 2, 6, 6, 4, 4, 5, 9, 7],
        [6, 0, 8, 3, 2, 6, 8, 4, 8, 8, 13, 7, 5, 8, 12, 10, 14],
        [9, 8, 0, 11, 10, 6, 3, 9, 5, 8, 4, 15, 14, 13, 9, 18, 9],
        [8, 3, 11, 0, 1, 7, 10, 6, 10, 10, 14, 6, 7, 9, 14, 6, 16],
        [7, 2, 10, 1, 0, 6, 9, 4, 8, 9, 13, 4, 6, 8, 12, 8, 14],
        [3, 6, 6, 7, 6, 0, 2, 3, 2, 2, 7, 9, 7, 7, 6, 12, 8],
        [6, 8, 3, 10, 9, 2, 0, 6, 2, 5, 4, 12, 10, 10, 6, 15, 5],
        [2, 4, 9, 6, 4, 3, 6, 0, 4, 4, 8, 5, 4, 3, 7, 8, 10],
        [3, 8, 5, 10, 8, 2, 2, 4, 0, 3, 4, 9, 8, 7, 3, 13, 6],
        [2, 8, 8, 10, 9, 2, 5, 4, 3, 0, 4, 6, 5, 4, 3, 9, 5],
        [6, 13, 4, 14, 13, 7, 4, 8, 4, 4, 0, 10, 9, 8, 4, 13, 4],
        [6, 7, 15, 6, 4, 9, 12, 5, 9, 6, 10, 0, 1, 3, 7, 3, 10],
        [4, 5, 14, 7, 6, 7, 10, 4, 8, 5, 9, 1, 0, 2, 6, 4, 8],
        [4, 8, 13, 9, 8, 7, 10, 3, 7, 4, 8, 3, 2, 0, 4, 5, 6],
        [5, 12, 9, 14, 12, 6, 6, 7, 3, 3, 4, 7, 6, 4, 0, 9, 2],
        [9, 10, 18, 6, 8, 12, 15, 8, 13, 9, 13, 3, 4, 5, 9, 0, 9],
        [7, 14, 9, 16, 14, 8, 5, 10, 6, 5, 4, 10, 8, 6, 2, 9, 0],
    ]
    data["time_windows"] = [
        (0, 5),  # depot
        (7, 12),  # 1
        (10, 15),  # 2
        (16, 18),  # 3
        (10, 13),  # 4
        (0, 5),  # 5
        (5, 10),  # 6
        (0, 4),  # 7
        (5, 10),  # 8
        (0, 3),  # 9
        (10, 16),  # 10
        (10, 15),  # 11
        (0, 5),  # 12
        (5, 10),  # 13
        (7, 8),  # 14
        (10, 15),  # 15
        (11, 15),  # 16
    ]
    data["num_vehicles"] = 4
    data["depot"] = 0
    return data


def print_solution(
    data: Dict[str, Any],
    manager: routing.IndexManager,
    routing_model: routing.Model,
    solution: constraint_solver.Assignment,
) -> None:
    """Prints solution on console."""
    print(f"Objective: {solution.objective_value()}")
    time_dimension = routing_model.get_dimension_or_die("Time")
    total_time = 0
    for vehicle_id in range(data["num_vehicles"]):
        if not routing_model.is_vehicle_used(solution, vehicle_id):
            continue
        index = routing_model.start(vehicle_id)
        plan_output = f"Route for vehicle {vehicle_id}:\n"
        while not routing_model.is_end(index):
            time_var = time_dimension.cumul_var(index)
            plan_output += (
                f"{manager.index_to_node(index)}"
                f" Time({solution.min(time_var)}, {solution.max(time_var)})"
                " -> "
            )
            index = solution.value(routing_model.next_var(index))
        time_var = time_dimension.cumul_var(index)
        plan_output += (
            f"{manager.index_to_node(index)}"
            f" Time({solution.min(time_var)},{solution.max(time_var)})\n"
        )
        plan_output += f"Time of the route: {solution.min(time_var)}min\n"
        print(plan_output)
        total_time += solution.min(time_var)
    print(f"Total time of all routes: {total_time}min")


def main() -> None:
    """Solve the VRP with time windows."""
    # Instantiate the data problem.
    data = create_data_model()

    # Create the routing index manager.
    manager = routing.IndexManager(
        len(data["time_matrix"]), data["num_vehicles"], data["depot"]
    )

    # Create Routing Model.
    routing_model = routing.Model(manager)

    # Create and register a transit callback.
    def time_callback(from_index: int, to_index: int) -> int:
        """Returns the travel time between the two nodes."""
        # Convert from routing variable Index to time matrix NodeIndex.
        from_node = manager.index_to_node(from_index)
        to_node = manager.index_to_node(to_index)
        return data["time_matrix"][from_node][to_node]

    transit_callback_index = routing_model.register_transit_callback(time_callback)

    # Define cost of each arc.
    routing_model.set_arc_cost_evaluator_of_all_vehicles(transit_callback_index)

    # Add Time Windows constraint.
    time = "Time"
    routing_model.add_dimension(
        transit_callback_index,
        30,  # allow waiting time
        30,  # maximum time per vehicle
        False,  # Don't force start cumul to zero.
        time,
    )
    time_dimension = routing_model.get_dimension_or_die(time)
    # Add time window constraints for each location except depot.
    for location_idx, time_window in enumerate(data["time_windows"]):
        if location_idx == data["depot"]:
            continue
        index = manager.node_to_index(location_idx)
        time_dimension.cumul_var(index).set_range(time_window[0], time_window[1])
    # Add time window constraints for each vehicle start node.
    depot_idx = data["depot"]
    for vehicle_id in range(data["num_vehicles"]):
        index = routing_model.start(vehicle_id)
        time_dimension.cumul_var(index).set_range(
            data["time_windows"][depot_idx][0], data["time_windows"][depot_idx][1]
        )

    # Instantiate route start and end times to produce feasible times.
    for i in range(data["num_vehicles"]):
        routing_model.add_variable_minimized_by_finalizer(
            time_dimension.cumul_var(routing_model.start(i))
        )
        routing_model.add_variable_minimized_by_finalizer(
            time_dimension.cumul_var(routing_model.end(i))
        )

    # Setting first solution heuristic.
    search_parameters: parameters_pb2.RoutingSearchParameters = (
        routing.default_routing_search_parameters()
    )
    search_parameters.first_solution_strategy = (
        enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    )

    # Solve the problem.
    solution = routing_model.solve_with_parameters(search_parameters)

    # Print solution on console.
    if solution:
        print_solution(data, manager, routing_model, solution)
    else:
        print("No solution found !")


main()

